<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## LSTM Bot QA

### Datos
El objecto es utilizar datos disponibles del challenge ConvAI2 (Conversational Intelligence Challenge 2) de conversaciones en inglés. Se construirá un BOT para responder a preguntas del usuario (QA).\
[LINK](http://convai.io/data/)

In [3]:
!pip install --upgrade --no-cache-dir gdown --quiet

In [4]:
import re



In [5]:
# Descargar la carpeta de dataset
import os
import gdown
if os.access('data_volunteers.json', os.F_OK) is False:
    url = 'https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN&export=download'
    output = 'data_volunteers.json'
    gdown.download(url, output, quiet=False)
else:
    print("El dataset ya se encuentra descargado")

Downloading...
From: https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN&export=download
To: /home/juan/CEIA/procesamiento_lenguaje_natural/clase_6/ejercicios/data_volunteers.json
100%|██████████| 2.58M/2.58M [00:00<00:00, 6.87MB/s]


In [6]:
# dataset_file
import json

text_file = "data_volunteers.json"
with open(text_file) as f:
    data = json.load(f) # la variable data será un diccionario



In [7]:
# Observar los campos disponibles en cada linea del dataset
data[0].keys()

dict_keys(['dialog', 'start_time', 'end_time', 'bot_profile', 'user_profile', 'eval_score', 'profile_match', 'participant1_id', 'participant2_id'])

In [13]:
# Observar el contenido de una linea del dataset
data[1]['dialog']

[{'id': 0,
  'sender': 'participant1',
  'text': 'Hello!',
  'evaluation_score': None,
  'sender_class': 'Human'},
 {'id': 1,
  'sender': 'participant2',
  'text': 'Hi! How are you?',
  'evaluation_score': None,
  'sender_class': 'Bot'},
 {'id': 2,
  'sender': 'participant1',
  'text': 'Not bad! And You?',
  'evaluation_score': None,
  'sender_class': 'Human'},
 {'id': 3,
  'sender': 'participant2',
  'text': "I'm doing well. Just got engaged to my high school sweetheart.",
  'evaluation_score': None,
  'sender_class': 'Bot'},
 {'id': 4,
  'sender': 'participant1',
  'text': 'Wowowowow! Congratulations! Is she pretty?',
  'evaluation_score': None,
  'sender_class': 'Human'},
 {'id': 5,
  'sender': 'participant2',
  'text': "She 's pretty cute. She invited me to dinner tonight. 🙂",
  'evaluation_score': None,
  'sender_class': 'Bot'},
 {'id': 6,
  'sender': 'participant1',
  'text': 'Cool! Have a good time you both! And what is your hobby?',
  'evaluation_score': None,
  'sender_class':

In [8]:
chat_in = []
chat_out = []

input_sentences = []
output_sentences = []
output_sentences_inputs = []
max_len = 30

def clean_text(txt):
    txt = txt.lower()    
    txt.replace("\'d", " had")
    txt.replace("\'s", " is")
    txt.replace("\'m", " am")
    txt.replace("don't", "do not")
    txt = re.sub(r'\W+', ' ', txt)
    
    return txt

for line in data:
    for i in range(len(line['dialog'])-1):
        # vamos separando el texto en "preguntas" (chat_in)
        # y "respuestas" (chat_out)
        chat_in = clean_text(line['dialog'][i]['text'])
        chat_out = clean_text(line['dialog'][i+1]['text'])

        if len(chat_in) >= max_len or len(chat_out) >= max_len:
            continue

        input_sentence, output = chat_in, chat_out
        
        # output sentence (decoder_output) tiene <eos>
        output_sentence = output + ' <eos>'
        # output sentence input (decoder_input) tiene <sos>
        output_sentence_input = '<sos> ' + output

        input_sentences.append(input_sentence)
        output_sentences.append(output_sentence)
        output_sentences_inputs.append(output_sentence_input)

print("Cantidad de rows utilizadas:", len(input_sentences))

Cantidad de rows utilizadas: 6033


In [9]:
input_sentences[1], output_sentences[1], output_sentences_inputs[1]

('hi how are you ', 'not bad and you  <eos>', '<sos> not bad and you ')

### 2 - Preprocesamiento
Realizar el preprocesamiento necesario para obtener:
- word2idx_inputs, max_input_len
- word2idx_outputs, max_out_len, num_words_output
- encoder_input_sequences, decoder_output_sequences, decoder_targets

### 3 - Preparar los embeddings
Utilizar los embeddings de Glove o FastText para transformar los tokens de entrada en vectores

### 4 - Entrenar el modelo
Entrenar un modelo basado en el esquema encoder-decoder utilizando los datos generados en los puntos anteriores. Utilce como referencias los ejemplos vistos en clase.

### 5 - Inferencia
Experimentar el funcionamiento de su modelo. Recuerde que debe realizar la inferencia de los modelos por separado de encoder y decoder.